# Encoder–HistGB

The baseline is the unchanged pipeline from **features_selection.ipynb**:
current imputation, engineered features and tuned HistGB, without feature selection.
We only append one contextual embedding per visit.

The encoder receives the original imputed features, their missingness indicators and elapsed
time since the first visit. It uses the confirmed best parameters from opt.ipynb:
three bidirectional Transformer encoder blocks, eight heads, width 32 and **no CLS token**. Each patient's available visits form one sequence.

Validation trains the whole pipeline separately in each patient split. There are **no
out-of-fold embeddings**. The outer validation patients never train the encoder or HistGB.
An internal patient holdout selects the epoch count; the encoder is then refitted on all
outer training patients for that many epochs.

## 1. Reuse the exact baseline and set training options

Only the five setup cells from features_selection.ipynb are executed, not its validation or
feature-selection experiments. Run this notebook from the project root or either notebook folder.
PyTorch is required: uv pip install --python .venv/bin/python torch.

In [1]:
import json
import os
from pathlib import Path
from copy import deepcopy
import torch
from torch import nn
from torch.nn.utils.rnn import pad_sequence
from torch.utils.data import DataLoader, TensorDataset
from sklearn.preprocessing import StandardScaler

PROJECT_DIR = next(p for p in (Path.cwd(), *Path.cwd().parents)
                   if (p / "exploration/features_selection.ipynb").is_file())
reference = json.loads((PROJECT_DIR / "exploration/features_selection.ipynb").read_text())
previous_dir = Path.cwd()
try:
    os.chdir(PROJECT_DIR)
    for i in [1, 3, 4, 6, 8]:  # Imports/data, history, imputation, features, pipeline.
        exec("".join(reference["cells"][i]["source"]))
finally:
    os.chdir(previous_dir)
DATA_DIR = PROJECT_DIR / "data"
baseline = clone(pipeline)

# Confirmed winner (trial 19) from opt.ipynb.
ENCODER = dict(width=32, heads=8, layers=3, dropout=0.1347886862599206,
               ff_multiplier=3, mlp_layers=2, mlp_units=16)
MAX_EPOCHS, PATIENCE, BATCH_SIZE = 100, 10, 128
LEARNING_RATE, WEIGHT_DECAY = 0.001759366313629433, 4.3514588969246546e-05
SEEDS = [42, 43, 44]
DEVICE = torch.device("cuda" if torch.cuda.is_available() else
                      "mps" if torch.backends.mps.is_available() else "cpu")
torch.set_num_threads(3)
print(f"Device: {DEVICE}; baseline: {baseline[-1]}")

Device: mps; baseline: HistGradientBoostingRegressor(l2_regularization=1.8115270802885095,
                              learning_rate=0.042592090635273853,
                              max_features=0.6078232265551354, max_iter=800,
                              max_leaf_nodes=62, min_samples_leaf=25,
                              random_state=42)


## 2. One sequence per patient

Patient IDs only group visits; they are never encoder inputs. Elapsed time encodes the actual
spacing between visits. Padding uses row number -1, which is ignored by attention and MSE.
The row numbers also restore embeddings to the original visit order after batching.

In [2]:
def encoder_features(imputed):
    features = imputed.drop(columns="patient_id").copy()
    features["time_since_first_visit"] = (
        imputed["age"] - imputed.groupby("patient_id")["age"].transform("min")
    )
    return features


def patient_sequences(raw, features, targets=None):
    visits = raw.reset_index(drop=True).sort_values(["patient_id", "age"])
    rows = [torch.tensor(group.index.to_numpy())
            for _, group in visits.groupby("patient_id", sort=False)]
    values = torch.tensor(np.asarray(features), dtype=torch.float32)
    labels = torch.tensor(np.zeros(len(raw)) if targets is None else np.asarray(targets),
                          dtype=torch.float32)
    return TensorDataset(
        pad_sequence([values[i] for i in rows], batch_first=True),
        pad_sequence([labels[i] for i in rows], batch_first=True),
        pad_sequence(rows, batch_first=True, padding_value=-1),
    )

## 3. Small bidirectional encoder

Dense input projection → three sequential TransformerEncoder layers → one embedding per visit.
Each layer already includes parallel attention heads, feed-forward layers, residual connections
and LayerNorm. The regression head is used for supervised training only.

In [3]:
class VisitEncoder(nn.Module):
    def __init__(self, n_features, width=64, heads=4, layers=2, dropout=0.1,
                 ff_multiplier=2, mlp_layers=0, mlp_units=32):
        super().__init__()
        self.projection = nn.Linear(n_features, width)
        layer = nn.TransformerEncoderLayer(
            d_model=width, nhead=heads, dim_feedforward=ff_multiplier * width,
            dropout=dropout, activation="gelu", batch_first=True,
        )
        self.encoder = nn.TransformerEncoder(layer, num_layers=layers,
                                             enable_nested_tensor=False)
        # TransformerEncoder copies its layer: give each block independent weights.
        for parameter in self.encoder.parameters():
            if parameter.ndim > 1:
                nn.init.xavier_uniform_(parameter)
        head, input_dim = [], width
        for _ in range(mlp_layers):
            head.extend([nn.Linear(input_dim, mlp_units), nn.GELU(), nn.Dropout(dropout)])
            input_dim = mlp_units
        self.head = nn.Sequential(*head, nn.Linear(input_dim, 1))

    def encode(self, values, padding):
        return self.encoder(self.projection(values), src_key_padding_mask=padding,
                            is_causal=False)

    def forward(self, values, padding):
        return self.head(self.encode(values, padding)).squeeze(-1)

## 4. Train the encoder inside a training split

Both input and target scaling are fitted on training patients only. Target scaling improves
optimization; reported predictions and MSE are converted back to OFF score units.
Early stopping uses its own patient holdout, including independently fitted preprocessing.
The selected epoch count is then used to refit a fresh encoder on all outer training patients.

In [4]:
def train_network(training, seed, epochs, validation=None):
    torch.manual_seed(seed)
    network = VisitEncoder(training.tensors[0].shape[-1], **ENCODER).to(DEVICE)
    optimizer = torch.optim.AdamW(network.parameters(), lr=LEARNING_RATE,
                                 weight_decay=WEIGHT_DECAY)
    loader = DataLoader(training, batch_size=BATCH_SIZE, shuffle=True)
    best_loss, best_epoch, best_state, stale = np.inf, epochs, None, 0
    history = []

    for epoch in range(1, epochs + 1):
        network.train()
        total_error, count = 0.0, 0
        for values, targets, rows in loader:
            values, targets, rows = (t.to(DEVICE) for t in (values, targets, rows))
            valid = rows >= 0
            optimizer.zero_grad()
            prediction = network(values, ~valid)
            loss = nn.functional.mse_loss(prediction[valid], targets[valid])
            loss.backward()
            nn.utils.clip_grad_norm_(network.parameters(), 1.0)
            optimizer.step()
            total_error += loss.item() * valid.sum().item()
            count += valid.sum().item()
        history.append({"epoch": epoch, "train_MSE": total_error / count})

        if validation is not None:
            network.eval()
            total_error, count = 0.0, 0
            with torch.no_grad():
                for values, targets, rows in DataLoader(validation, batch_size=BATCH_SIZE):
                    values, targets, rows = (t.to(DEVICE) for t in (values, targets, rows))
                    valid = rows >= 0
                    prediction = network(values, ~valid)
                    total_error += ((prediction[valid] - targets[valid]) ** 2).sum().item()
                    count += valid.sum().item()
            stop_loss = total_error / count
            history[-1]["stop_MSE"] = stop_loss
            if stop_loss < best_loss:
                best_loss, best_epoch, stale = stop_loss, epoch, 0
                best_state = deepcopy(network.state_dict())
            else:
                stale += 1
            if stale >= PATIENCE:
                break

    if best_state is not None:
        network.load_state_dict(best_state)
    return network.eval(), best_epoch, pd.DataFrame(history)


def fit_encoder(raw, targets, imputed, seed):
    fit_idx, stop_idx = next(GroupShuffleSplit(
        n_splits=1, test_size=0.2, random_state=seed
    ).split(raw, groups=raw["patient_id"]))
    fit_raw, stop_raw = raw.iloc[fit_idx], raw.iloc[stop_idx]
    inner_prep = clone(baseline[:2])
    fit_features = encoder_features(inner_prep.fit_transform(fit_raw))
    stop_features = encoder_features(inner_prep.transform(stop_raw))
    scaler = StandardScaler().fit(fit_features)
    mean = targets.iloc[fit_idx].mean()
    scale = max(targets.iloc[fit_idx].std(ddof=0), 1e-8)
    training = patient_sequences(fit_raw, scaler.transform(fit_features),
                                 (targets.iloc[fit_idx] - mean) / scale)
    stopping = patient_sequences(stop_raw, scaler.transform(stop_features),
                                 (targets.iloc[stop_idx] - mean) / scale)
    _, epochs, history = train_network(training, seed, MAX_EPOCHS, stopping)
    history[["train_MSE", "stop_MSE"]] *= scale ** 2

    # Fresh model and scaling, now fitted on every patient in the outer training split.
    features = encoder_features(imputed)
    scaler = StandardScaler().fit(features)
    mean, scale = targets.mean(), max(targets.std(ddof=0), 1e-8)
    training = patient_sequences(raw, scaler.transform(features), (targets - mean) / scale)
    network, _, _ = train_network(training, seed, epochs)
    network.requires_grad_(False)
    return dict(network=network, scaler=scaler, target_mean=mean, target_scale=scale,
                epochs=epochs, history=history)

## 5. Extract frozen embeddings in the original row order

The prediction head stays available for diagnostics; HistGB receives only the contextual
embeddings, with no CLS token, pooling or predicted-score column.

In [5]:
@torch.no_grad()
def encoder_outputs(fitted, raw, imputed):
    features = fitted["scaler"].transform(encoder_features(imputed))
    sequences = patient_sequences(raw, features)
    network = fitted["network"].eval()
    embeddings = np.empty((len(raw), network.projection.out_features), dtype=np.float32)
    predictions = np.empty(len(raw), dtype=np.float32)
    for values, _, rows in DataLoader(sequences, batch_size=BATCH_SIZE):
        padding = rows < 0
        hidden = network.encode(values.to(DEVICE), padding.to(DEVICE))
        prediction = network.head(hidden).squeeze(-1).cpu()
        valid_rows = rows[~padding].numpy()
        embeddings[valid_rows] = hidden.cpu()[~padding].numpy()
        predictions[valid_rows] = prediction[~padding].numpy()
    embeddings = pd.DataFrame(embeddings, index=raw.index,
                              columns=[f"enc_{i:02d}" for i in range(embeddings.shape[1])])
    predictions = predictions * fitted["target_scale"] + fitted["target_mean"]
    return embeddings, pd.Series(predictions, index=raw.index)

## 6. Fold-specific Encoder–HistGB validation

Use the same five patient-separated splits as the existing notebooks. Preprocessing is fitted
again in each split and shared by the unchanged baseline and augmented model. For each seed,
train a fresh encoder on that split's training patients, freeze it, append its embeddings and
fit a fresh clone of the exact same HistGB. No embeddings are combined across splits.
Negative delta_MSE means the embeddings help. The standard deviation describes variability across splits and seeds, not uncertainty about the true test performance.

In [6]:
cv = GroupShuffleSplit(n_splits=5, test_size=0.2, random_state=42)
scores = []
tuning_folds = []
with threadpool_limits(limits=3):
    for fold, (train_idx, val_idx) in enumerate(cv.split(X, y, groups=X["patient_id"]), 1):
        raw_train, raw_val = X.iloc[train_idx], X.iloc[val_idx]
        yt, yv = y.iloc[train_idx], y.iloc[val_idx]
        prep = clone(baseline[:-1])
        train = prep.fit_transform(raw_train, yt)
        validation = prep.transform(raw_val)
        imputed_train = prep[:2].transform(raw_train)
        imputed_val = prep[:2].transform(raw_val)
        original = clone(baseline[-1]).fit(train, yt)
        baseline_mse = mean_squared_error(yv, original.predict(validation))

        for seed in SEEDS:
            fitted_encoder = fit_encoder(raw_train, yt, imputed_train, seed)
            z_train, _ = encoder_outputs(fitted_encoder, raw_train, imputed_train)
            z_val, encoder_prediction = encoder_outputs(fitted_encoder, raw_val, imputed_val)
            tuning_folds.append((train.join(z_train), validation.join(z_val), yt, yv))
            augmented = clone(baseline[-1]).fit(train.join(z_train), yt)
            mse = mean_squared_error(yv, augmented.predict(validation.join(z_val)))
            scores.append(dict(fold=fold, seed=seed, epochs=fitted_encoder["epochs"],
                               HistGB=baseline_mse, Transformer=mean_squared_error(yv, encoder_prediction),
                               Encoder_HistGB=mse, delta_MSE=mse - baseline_mse))
            print(f"Fold {fold}/5, seed {seed}, {fitted_encoder['epochs']} epochs: "
                  f"HistGB {baseline_mse:.4f} → Encoder–HistGB {mse:.4f}")
scores = pd.DataFrame(scores)
scores

Fold 1/5, seed 42, 56 epochs: HistGB 12.1961 → Encoder–HistGB 7.5995
Fold 1/5, seed 43, 79 epochs: HistGB 12.1961 → Encoder–HistGB 7.1951
Fold 1/5, seed 44, 66 epochs: HistGB 12.1961 → Encoder–HistGB 7.3864
Fold 2/5, seed 42, 67 epochs: HistGB 11.5942 → Encoder–HistGB 7.5481
Fold 2/5, seed 43, 77 epochs: HistGB 11.5942 → Encoder–HistGB 7.1661
Fold 2/5, seed 44, 49 epochs: HistGB 11.5942 → Encoder–HistGB 7.5765
Fold 3/5, seed 42, 86 epochs: HistGB 11.5700 → Encoder–HistGB 6.8773
Fold 3/5, seed 43, 37 epochs: HistGB 11.5700 → Encoder–HistGB 7.7778
Fold 3/5, seed 44, 18 epochs: HistGB 11.5700 → Encoder–HistGB 8.2516
Fold 4/5, seed 42, 69 epochs: HistGB 11.7891 → Encoder–HistGB 7.0266
Fold 4/5, seed 43, 27 epochs: HistGB 11.7891 → Encoder–HistGB 7.8758
Fold 4/5, seed 44, 52 epochs: HistGB 11.7891 → Encoder–HistGB 7.2814
Fold 5/5, seed 42, 60 epochs: HistGB 12.4677 → Encoder–HistGB 7.5157
Fold 5/5, seed 43, 59 epochs: HistGB 12.4677 → Encoder–HistGB 7.6556
Fold 5/5, seed 44, 72 epochs: Hist

,fold,seed,epochs,HistGB,Transformer,Encoder_HistGB,delta_MSE
0,1,42,56,12.196120,8.763222,7.599489,-4.596632
1,1,43,79,12.196120,7.725109,7.195100,-5.001020
2,1,44,66,12.196120,7.772806,7.386380,-4.809740
3,2,42,67,11.594218,8.138651,7.548126,-4.046092
4,2,43,77,11.594218,7.860277,7.166142,-4.428076
5,2,44,49,11.594218,8.584432,7.576550,-4.017668
6,3,42,86,11.569963,7.411742,6.877261,-4.692702
7,3,43,37,11.569963,9.355146,7.777821,-3.792142
8,3,44,18,11.569963,9.650412,8.251584,-3.318380
9,4,42,69,11.789132,8.674991,7.026603,-4.762528


In [7]:
summary = scores[["HistGB", "Transformer", "Encoder_HistGB"]].agg(
    ["mean", lambda s: s.std(ddof=0)]
).T
summary.columns = ["MSE", "std"]
summary["RMSE"] = np.sqrt(summary["MSE"])
summary["gain_MSE_pct"] = 100 * (1 - summary["MSE"] / summary.loc["HistGB", "MSE"])
summary

,MSE,std,RMSE,gain_MSE_pct
HistGB,11.923423,0.352683,3.453031,0.000000
Transformer,8.429465,0.794165,2.903354,29.303310
Encoder_HistGB,7.477092,0.336742,2.734427,37.290727


## 7. Re-optimize HistGB with the fixed winning encoder

Reuse the frozen embeddings from the five patient splits and three seeds above.
As in features_engineering.ipynb, run 40 Optuna TPE trials with median pruning,
starting with the current HistGB parameters. Only HistGB is retrained during search.
The comparison below is tuning validation; final training uses best_histgb.


In [9]:
import optuna
from threadpoolctl import threadpool_limits

N_TRIALS = 40
optuna.logging.set_verbosity(optuna.logging.INFO)

# Reuse the frozen embeddings cached above for all five splits and three seeds.
def histgb_objective(trial):
    params = {
        "learning_rate": trial.suggest_float("learning_rate", 0.02, 0.2, log=True),
        "max_iter": trial.suggest_int("max_iter", 100, 800, step=100),
        "max_leaf_nodes": trial.suggest_int("max_leaf_nodes", 7, 63, log=True),
        "min_samples_leaf": trial.suggest_int("min_samples_leaf", 10, 120, log=True),
        "l2_regularization": trial.suggest_float("l2_regularization", 0, 30),
        "max_features": trial.suggest_float("max_features", 0.6, 1.0),
    }
    scores = []
    for step, (train, validation, yt, yv) in enumerate(tuning_folds):
        fitted = clone(baseline[-1]).set_params(**params).fit(train, yt)
        scores.append(mean_squared_error(yv, fitted.predict(validation)))
        # Report after all seeds of a patient split; prune after at least two splits.
        if (step + 1) % len(SEEDS) == 0:
            trial.report(float(np.mean(scores)), step // len(SEEDS))
            if step < len(tuning_folds) - 1 and trial.should_prune():
                raise optuna.TrialPruned()
    trial.set_user_attr("fold_mse", scores)
    return float(np.mean(scores))


histgb_study = optuna.create_study(
    direction="minimize", sampler=optuna.samplers.TPESampler(seed=42),
    pruner=optuna.pruners.MedianPruner(n_startup_trials=8, n_warmup_steps=1),
)
parameter_names = ["learning_rate", "max_iter", "max_leaf_nodes", "min_samples_leaf", "l2_regularization", "max_features"]
histgb_study.enqueue_trial({name: baseline[-1].get_params()[name] for name in parameter_names})
with threadpool_limits(limits=3):
    histgb_study.optimize(histgb_objective, n_trials=N_TRIALS)

baseline_scores = histgb_study.trials[0].user_attrs["fold_mse"]
best_scores = histgb_study.best_trial.user_attrs["fold_mse"]
tuning_summary = pd.DataFrame([
    {"model": name, "MSE": np.mean(scores), "std": np.std(scores)}
    for name, scores in [("Encoder_HistGB_current", baseline_scores), ("Encoder_HistGB_tuned", best_scores)]
]).set_index("model")
tuning_summary["RMSE"] = np.sqrt(tuning_summary["MSE"])
tuning_summary["gain_MSE_pct"] = 100 * (1 - tuning_summary["MSE"] / np.mean(baseline_scores))
best_histgb = clone(baseline[-1]).set_params(**histgb_study.best_params)
print("Best HistGB parameters:", histgb_study.best_params)
tuning_summary


[I 2026-11-02 10:13:01,230] A new study created in memory with name: no-name-19bef18c-b9db-4ea5-aa47-70647dd94ef3
[I 2026-11-02 10:15:07,092] Trial 0 finished with value: 7.4770919593685825 and parameters: {'learning_rate': 0.042592090635273853, 'max_iter': 800, 'max_leaf_nodes': 62, 'min_samples_leaf': 25, 'l2_regularization': 1.8115270802885095, 'max_features': 0.6078232265551354}. Best is trial 0 with value: 7.4770919593685825.
[I 2026-11-02 10:16:44,175] Trial 1 finished with value: 7.535900962584248 and parameters: {'learning_rate': 0.047377279007281566, 'max_iter': 800, 'max_leaf_nodes': 34, 'min_samples_leaf': 43, 'l2_regularization': 4.680559213273096, 'max_features': 0.662397808134481}. Best is trial 0 with value: 7.4770919593685825.
[I 2026-11-02 10:18:20,698] Trial 2 finished with value: 7.613700048133574 and parameters: {'learning_rate': 0.022861967752626437, 'max_iter': 700, 'max_leaf_nodes': 26, 'min_samples_leaf': 57, 'l2_regularization': 0.6175348288740734, 'max_feature

Best HistGB parameters: {'learning_rate': 0.03471523676107423, 'max_iter': 800, 'max_leaf_nodes': 62, 'min_samples_leaf': 16, 'l2_regularization': 8.305016398991823, 'max_features': 0.6408427464001498}


,MSE,std,RMSE,gain_MSE_pct
model,,,,
Encoder_HistGB_current,7.477092,0.336742,2.734427,0.000000
Encoder_HistGB_tuned,7.452323,0.333780,2.729894,0.331264


## 8. Optional final training and test predictions

After reviewing validation, set RUN_FINAL to True and choose a seed. A single encoder is
trained on all training patients and paired with a single HistGB using the best parameters from the HistGB search above.
The outer test data is transformed only; it never fits preprocessing or either model.

In [ ]:
RUN_FINAL = False
FINAL_SEED = 43

if RUN_FINAL:
    X_test = pd.read_csv(DATA_DIR / "X_test_oiZ2ukx.csv")[X.columns]
    with threadpool_limits(limits=3):
        final_prep = clone(baseline[:-1])
        train = final_prep.fit_transform(X, y)
        test = final_prep.transform(X_test)
        imputed_train = final_prep[:2].transform(X)
        imputed_test = final_prep[:2].transform(X_test)
        final_encoder = fit_encoder(X, y, imputed_train, FINAL_SEED)
        z_train, _ = encoder_outputs(final_encoder, X, imputed_train)
        z_test, _ = encoder_outputs(final_encoder, X_test, imputed_test)
        final_histgb = clone(best_histgb).fit(train.join(z_train), y)
        prediction = final_histgb.predict(test.join(z_test))
    assert np.isfinite(prediction).all()
    submission = pd.DataFrame({"Index": X_test["Index"], "target": prediction})
    submission_path = DATA_DIR / "submission_encoder_histgb_retuned.csv"
    submission.to_csv(submission_path, index=False)
    print(f"Saved {submission_path} ({len(submission):,} visits)")

## Alternatives to test later

- Add the existing engineered features to encoder inputs.
- Vary encoder depth, width and heads, or replace the encoder architecture.
- Predict ON/OFF measurements with this encoder or a separate encoder.
- Feed HistGB the encoder's final prediction instead of, or alongside, its embeddings.